# Veri Bilimi Ara Ödev
Bu notebook yerel/Colab uyumludur. Eksik sütun üretmez; şema uyumsuzsa ilgili soruları **blocked** gösterir.

## Kurulum ve veri yolu
Colab'da dosya panelinden CSV yükleyin veya Drive bağlayın. Opsiyonel Kaggle indirme komutu (credentials kullanıcı tarafından sağlanır): `!kaggle datasets download -d thetrumpet/teknofest-trendyol-2026-datathonn --unzip -p data`.

In [ ]:
from pathlib import Path
import json, random, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42); np.random.seed(42)
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebook': PROJECT_DIR = PROJECT_DIR.parent
DATA_PATH = Path('data/items.csv')  # Colab/yerel için değiştirilebilir
OUTPUT_DIR = PROJECT_DIR / 'outputs'; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = ['indirim_orani','musteri_puani','odeme_turu','musteri_tipi','siparis_tarihi','sehir','kategori','birim_fiyat','toplam_tutar','teslimat_gunu']
if DATA_PATH.is_file():
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.DataFrame()
missing_required = [c for c in REQUIRED if c not in df.columns]
COMPATIBLE = bool(len(df)) and not missing_required
print('Veri durumu:', 'Uyumlu' if COMPATIBLE else 'BLOCKED')
print('Eksik zorunlu sütunlar:', missing_required)
df_original = df.copy(deep=True)
df_clean = df.copy(deep=True)

## 1. İlk 10 ve son 10 satır
Verinin başlangıç ve bitiş kayıtları birlikte incelenir.

In [ ]:
display(df_original.head(10))
display(df_original.tail(10))
print('Yorum: Kayıt örnekleri veri düzeninin ilk kontrolünü sağlar.' if len(df_original) else 'BLOCKED: Yerel veri yok.')

## 2. Satır, sütun sayısı ve sütun adları
Veri setinin temel boyutları raporlanır.

In [ ]:
print('Satır:', df_original.shape[0], 'Sütun:', df_original.shape[1])
print('Sütunlar:', list(df_original.columns))
print('Yorum: Boyut bilgisi analiz kapsamını gösterir.')

## 3. info(), describe(), sayısal ve kategorik sütunlar
Tipler ve betimleyici istatistikler ayrıştırılır.

In [ ]:
df_original.info()
display(df_original.describe(include='all').T if len(df_original.columns) else pd.DataFrame())
numeric_columns = df_original.select_dtypes(include='number').columns.tolist()
categorical_columns = df_original.select_dtypes(exclude='number').columns.tolist()
print('Sayısal:', numeric_columns); print('Kategorik:', categorical_columns)
print('Yorum: Tip listeleri sonraki temizleme adımlarını yönlendirir.')

## 4. Eksik sayı/yüzde ve ilk beş sütun
Eksik değer öncelikleri hesaplanır.

In [ ]:
missing = pd.DataFrame({'missing_count':df_original.isna().sum(), 'missing_percentage':df_original.isna().mean()*100}).sort_values('missing_count', ascending=False)
display(missing.head())
missing.to_csv(OUTPUT_DIR/'missing_values.csv')
print('Yorum: En yüksek eksiklik oranına sahip alanlar önce ele alınmalıdır.')

## 5. En az bir eksik içeren ilk 20 satır
Eksiklik örüntüleri satır düzeyinde görülür.

In [ ]:
display(df_original[df_original.isna().any(axis=1)].head(20) if len(df_original.columns) else pd.DataFrame())
print('Yorum: Birlikte eksilen alanlar veri toplama sorunlarına işaret edebilir.')

## 6. indirim_orani ve musteri_puani median imputasyonu
Sayısal eksikler robust median ile doldurulur; sütun yoksa işlem engellenir.

In [ ]:
if COMPATIBLE:
    for c in ['indirim_orani','musteri_puani']:
        df_clean[c] = pd.to_numeric(df_clean[c], errors='coerce')
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())
    display(df_clean[['indirim_orani','musteri_puani']].head())
    print('Yorum: Median uç değerlere ortalamadan daha dayanıklıdır.')
else: print('BLOCKED:', missing_required)

## 7. odeme_turu ve musteri_tipi mode imputasyonu
Kategorik eksikler en sık sınıfla doldurulur.

In [ ]:
if COMPATIBLE:
    for c in ['odeme_turu','musteri_tipi']:
        mode = df_clean[c].mode(dropna=True)
        if not mode.empty: df_clean[c] = df_clean[c].fillna(mode.iloc[0])
    display(df_clean[['odeme_turu','musteri_tipi']].head())
    print('Yorum: Mode mevcut kategori sözlüğünü korur.')
else: print('BLOCKED:', missing_required)

## 8. Tarih dönüşümü ve tarih özellikleri
Sipariş tarihi datetime'a çevrilerek yıl, ay, gün ve hafta günü üretilir.

In [ ]:
if COMPATIBLE:
    df_clean['siparis_tarihi'] = pd.to_datetime(df_clean['siparis_tarihi'], errors='coerce')
    df_clean['siparis_yili'] = df_clean['siparis_tarihi'].dt.year
    df_clean['siparis_ayi'] = df_clean['siparis_tarihi'].dt.month
    df_clean['siparis_gunu'] = df_clean['siparis_tarihi'].dt.day
    df_clean['haftanin_gunu'] = df_clean['siparis_tarihi'].dt.day_name()
    display(df_clean[['siparis_tarihi','siparis_yili','siparis_ayi','siparis_gunu','haftanin_gunu']].head())
    print('Yorum: Tarih bileşenleri zamansal analiz için hazırdır.')
else: print('BLOCKED:', missing_required)

## 9. Duplicate sayısı ve kaldırma
Orijinal veri korunarak temiz kopyadaki tekrarlar kaldırılır.

In [ ]:
duplicate_count = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().copy()
print('Duplicate:', duplicate_count, 'Kalan:', len(df_clean))
(OUTPUT_DIR/'duplicate_summary.json').write_text(json.dumps({'duplicate_rows':duplicate_count}, indent=2))
print('Yorum: Orijinal dataframe değiştirilmedi.')

## 10. Şehir yazımlarını normalize etme
Boşluk ve büyük/küçük harf farklılıkları normalize edilir.

In [ ]:
if COMPATIBLE:
    df_clean['sehir'] = df_clean['sehir'].astype('string').str.strip().str.title().str.replace(r'\s+',' ',regex=True)
    display(df_clean['sehir'].value_counts().head())
    print('Yorum: Yalnızca yazım biçimi normalize edildi; şehir uydurulmadı.')
else: print('BLOCKED:', missing_required)

## 11. Kategori yazımlarını normalize etme
Kategori etiketlerine aynı güvenli metin normalizasyonu uygulanır.

In [ ]:
if COMPATIBLE:
    df_clean['kategori'] = df_clean['kategori'].astype('string').str.strip().str.title().str.replace(r'\s+',' ',regex=True)
    display(df_clean['kategori'].value_counts().head())
    print('Yorum: Semantik kategori eşleştirmesi veri sözlüğü olmadan yapılmadı.')
else: print('BLOCKED:', missing_required)

## 12. Mantıksal olarak geçersiz kayıtlar
Negatif/sıfır tutarlar, negatif teslimat ve 5 üzeri puan temiz kopyadan kaldırılır.

In [ ]:
if COMPATIBLE:
    for c in ['birim_fiyat','toplam_tutar','teslimat_gunu','musteri_puani']: df_clean[c]=pd.to_numeric(df_clean[c],errors='coerce')
    invalid = (df_clean['birim_fiyat']<=0)|(df_clean['toplam_tutar']<=0)|(df_clean['teslimat_gunu']<0)|(df_clean['musteri_puani']>5)
    invalid_count=int(invalid.sum()); df_clean=df_clean.loc[~invalid].copy()
    print('Silinen geçersiz kayıt:', invalid_count)
    print('Yorum: Kurallar açıkça tanımlanan mantıksal sınırlarla sınırlıdır.')
else: print('BLOCKED:', missing_required)

## 13. birim_fiyat IQR analizi
Aykırı değer sınırları 1.5×IQR ile raporlanır; otomatik silme yapılmaz.

In [ ]:
if COMPATIBLE:
    s=df_clean['birim_fiyat'].dropna(); q1,q3=s.quantile([.25,.75]); iqr=q3-q1; lo,hi=q1-1.5*iqr,q3+1.5*iqr
    outlier=pd.DataFrame([{'q1':q1,'q3':q3,'iqr':iqr,'lower':lo,'upper':hi,'count':int(((s<lo)|(s>hi)).sum())}])
    display(outlier); outlier.to_csv(OUTPUT_DIR/'outlier_analysis.csv',index=False)
    print('Yorum: Aykırılar meşru ürün fiyatları olabileceği için yalnızca raporlandı.')
else: print('BLOCKED:', missing_required)

## 14. toplam_tutar histogram ve box plot
Dağılım şekli ve uç değerler iki tamamlayıcı grafikle incelenir.

In [ ]:
if COMPATIBLE:
    fig,axes=plt.subplots(1,2,figsize=(12,4)); df_clean['toplam_tutar'].hist(bins=30,ax=axes[0]); axes[0].set_title('Toplam Tutar Histogram'); axes[1].boxplot(df_clean['toplam_tutar'].dropna(),vert=False); axes[1].set_title('Toplam Tutar Box Plot'); plt.tight_layout(); plt.show()
    print('Yorum: Histogram dağılımı, box plot aykırı gözlemleri gösterir.')
else: print('BLOCKED:', missing_required)

## 15. Kategorik ve sayısal analizler
Kategori, müşteri tipi, ödeme türü ve sayısal özetler birlikte değerlendirilir.

In [ ]:
if COMPATIBLE:
    display(df_clean['kategori'].value_counts())
    display(df_clean['musteri_tipi'].value_counts(normalize=True).mul(100))
    df_clean['odeme_turu'].value_counts().plot(kind='bar',title='Ödeme Türü'); plt.show()
    display(df_clean.describe())
    category_summary=df_clean.groupby('kategori')['toplam_tutar'].mean().sort_values(ascending=False)
    customer_summary=df_clean.groupby('musteri_tipi')['musteri_puani'].mean().sort_values(ascending=False)
    display(category_summary.head()); display(customer_summary.head())
    category_summary.to_csv(OUTPUT_DIR/'category_summary.csv'); customer_summary.to_csv(OUTPUT_DIR/'customer_type_summary.csv')
    df_clean.to_csv(OUTPUT_DIR/'final_clean_dataset.csv',index=False)
    print('Yorum: Sonuçlar ortalama ve sınıf büyüklükleri birlikte okunmalıdır.')
else: print('BLOCKED:', missing_required)

## Tekrarlanabilir çıktıların kaydedilmesi
Uyumlu veri varsa zorunlu rapor ve görseller yerel `outputs/` dizinine yazılır.

In [ ]:
if COMPATIBLE:
    pd.DataFrame({'metric':['rows','columns'],'value':[len(df_original),len(df_original.columns)]}).to_csv(OUTPUT_DIR/'data_profile.csv',index=False)
    cleaning_summary={'input_rows':len(df_original),'duplicate_rows':duplicate_count,'invalid_rows_removed':invalid_count,'output_rows':len(df_clean)}
    (OUTPUT_DIR/'cleaning_summary.json').write_text(json.dumps(cleaning_summary,indent=2))
    fig=df_clean['toplam_tutar'].plot.hist(bins=30,title='Toplam Tutar').get_figure(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'total_amount_histogram.png',dpi=180); plt.close(fig)
    fig,ax=plt.subplots(figsize=(8,3)); ax.boxplot(df_clean['toplam_tutar'].dropna(),vert=False); ax.set_title('Toplam Tutar'); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'total_amount_boxplot.png',dpi=180); plt.close(fig)
    fig=df_clean['odeme_turu'].value_counts().plot.bar(title='Ödeme Türü').get_figure(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/'payment_type_countplot.png',dpi=180); plt.close(fig)
    (OUTPUT_DIR/'final_summary.md').write_text(f'# Final Özet\n\n- Girdi: {len(df_original)}\n- Temiz: {len(df_clean)}\n')
    print('Tüm zorunlu çıktılar kaydedildi:', OUTPUT_DIR)
else: print('BLOCKED: Veri/şema uyumlu olmadığı için sahte çıktı üretilmedi.')